
# Wine Reviews — Data Acquisition & Preprocessing (PySpark)

**Role:** Data acquisition & preprocessing lead — *Kevaughna McClymont* & *Ce Liang*  

This notebook covers: data acquisition (optional download from Kaggle), data reading and type correction, deduplication, string normalization, handling of missing and abnormal values, text cleaning and tokenization of descriptions, TF-IDF text feature extraction, structured feature encoding (e.g., Country/Variety), and saving of the cleaned and model-ready datasets / preprocessing models.

> Recommended Environment: Spark 3.4+ / 3.5+, Python 3.9+.
This notebook can run in local local[*] mode or on a cluster.
If using Colab, please enable a high-memory runtime and install pyspark first.


## 0. Dependency Installation (Only Required for First Run)

> If your environment already has PySpark installed, you can skip this section.  
> For Colab / local environments, simply execute:


In [1]:

# Optional: Install dependencies (skip if already installed)
# In some managed environments, it’s recommended to pin versions to avoid compatibility issues.

try:
    import pyspark  # noqa
    print("PySpark is already installed.")
except Exception:
    %pip install -q pyspark==3.5.1

# 可选：文本规范化
try:
    import unidecode  # noqa
except Exception:
    %pip install -q Unidecode


PySpark is already installed.
Note: you may need to restart the kernel to use updated packages.


### (Optional) Download Data from Kaggle
If you want to download directly from Kaggle (dataset: `zynicide/wine-reviews`), please:

1. Place your Kaggle API Token in `~/.kaggle/kaggle.json` on your local machine;  
2. Run the code below to automatically download and extract the dataset;  
   or manually place `winemag-data-130k-v2.csv` into the `data/` directory.


In [ ]:

# Optional: Download using Kaggle API (skip if CSV has already been prepared manually)

import os, zipfile, pathlib, subprocess, sys

DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)

def have_kaggle():
    return (pathlib.Path("~/.kaggle/kaggle.json").expanduser().exists()
            or pathlib.Path("/root/.kaggle/kaggle.json").exists())

if have_kaggle():
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
        cmd = ["kaggle", "datasets", "download", "-d", "zynicide/wine-reviews", "-p", str(DATA_DIR)]
        print(" ".join(cmd))
        subprocess.run(cmd, check=True)
        # 解压最新 zip
        for z in DATA_DIR.glob("*.zip"):
            with zipfile.ZipFile(z) as f:
                f.extractall(DATA_DIR)
        print("Kaggle dataset downloaded to:", DATA_DIR.resolve())
    except Exception as e:
        print("Kaggle download failed (you can ignore this if you already have the CSV):", e)
else:
    print("Kaggle token not found; skip download. Place CSV under ./data/ manually.")


## 1. Start Spark Session and Global Configuration


In [65]:

import os
from pyspark.sql import SparkSession

APP_NAME = "Wine-Reviews-Preprocess"
MASTER = os.environ.get("SPARK_MASTER", "local[*]")

spark = (
    SparkSession.builder
    .appName(APP_NAME)
    .master(MASTER)
    #.config("spark.master", "local[*]")
    #.config("spark.driver.memory", "12g")
    #.config("spark.sql.execution.arrow.pyspark.enabled", "true")
    #.config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    #.config("spark.ui.showConsoleProgress", "true")
    .config("spark.driver.memory", "16g")
    .config("spark.executor.memory", "12g")
    .config("spark.memory.fraction", "0.85")          # Allocate more heap memory for execution
    .config("spark.memory.storageFraction", "0.3")    # Reduce cache memory fraction
    # Performance and fault tolerance
    .config("spark.sql.shuffle.partitions", "200")    # Avoid oversized single partitions
    .config("spark.sql.autoBroadcastJoinThreshold", -1) # Disable large table broadcast
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)
spark.conf.set("spark.sql.legacy.OneHotEncoder.vectorOutput", "true")

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Master:", MASTER)


Spark version: 3.5.5
Master: local[*]


## 2. Data Source Path Configuration (Auto-Detect Common Paths)

- By default, the script will attempt to locate the dataset in the following paths:
  - `./data/winemag-data-130k-v2.csv`
  - `./winemag-data-130k-v2.csv`
  - `./data/winemag-data_first150k.csv`
- You can also specify a custom path via the environment variable `WINE_CSV_PATH`.


In [66]:

import pathlib

DEFAULT_PATHS = [
    "data/winemag-data-130k-v2.csv",
    "winemag-data-130k-v2.csv",
    "data/winemag-data_first150k.csv",
    "winemag-data_first150k.csv",
    # try at same time：
    "/mnt/data/winemag-data-130k-v2.csv",
    "/mnt/data/winemag-data_first150k.csv",
]

input_path = os.environ.get("WINE_CSV_PATH")
if not input_path:
    for p in DEFAULT_PATHS:
        if pathlib.Path(p).exists():
            input_path = p
            break

assert input_path and pathlib.Path(input_path).exists(), (
    "didnt find data files，please set env viariable WINE_CSV_PATH or put CSV under ./data/ "
)

OUTPUT_DIR = pathlib.Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

input_path


'data/winemag-data-130k-v2.csv'

## 3. Read Data and Perform Initial Inspection (Infer / Correct Data Types)


In [67]:

from pyspark.sql import functions as F, types as T

df_raw = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(input_path, inferSchema=True)
)

print("Raw count:", df_raw.count())
print("Columns:", df_raw.columns)

# Kaggle WineEnthusiast：
expected_cols = [
    "country","description","designation","points","price","province",
    "region_1","region_2","taster_name","taster_twitter_handle",
    "title","variety","winery"
]

# for ex. 'Unnamed: 0'
for junk in ["Unnamed: 0", "_c0", "index"]:
    if junk in df_raw.columns:
        df_raw = df_raw.drop(junk)

# fix type
df = (
    df_raw
    .withColumn("points", F.col("points").cast(T.IntegerType()))
    .withColumn("price", F.regexp_replace(F.col("price").cast(T.StringType()), ",", ""))
    .withColumn("price", F.col("price").cast(T.DoubleType()))
)

df.printSchema()
df.show(5, truncate=80)


25/11/09 14:24:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv


Raw count: 129971
Columns: ['_c0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']
root
 |-- country: string (nullable = true)
 |-- description: string (nullable = true)
 |-- designation: string (nullable = true)
 |-- points: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- province: string (nullable = true)
 |-- region_1: string (nullable = true)
 |-- region_2: string (nullable = true)
 |-- taster_name: string (nullable = true)
 |-- taster_twitter_handle: string (nullable = true)
 |-- title: string (nullable = true)
 |-- variety: string (nullable = true)
 |-- winery: string (nullable = true)

+--------+--------------------------------------------------------------------------------+----------------------------------+------+-----+-----------------+-------------------+-----------------+------------------+---------------------+--------------------------

25/11/09 14:24:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv


## 4. Quick Data Quality Check & Utility Functions


In [68]:

from typing import List

def null_report(_df):
    total = _df.count()
    stats = [
        F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), c)).alias(c)
        for c in _df.columns
    ]
    row = _df.select(*stats).collect()[0].asDict()
    return sorted([(k, v, round(v/total, 4)) for k, v in row.items()], key=lambda x: x[1], reverse=True)

null_report(df)[:10]


25/11/09 14:24:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:24:43 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv


[('region_2', 79460, 0.6114),
 ('designation', 37465, 0.2883),
 ('taster_twitter_handle', 31213, 0.2402),
 ('taster_name', 26244, 0.2019),
 ('region_1', 21247, 0.1635),
 ('price', 8996, 0.0692),
 ('country', 63, 0.0005),
 ('province', 63, 0.0005),
 ('variety', 1, 0.0),
 ('description', 0, 0.0)]

## 5. String Normalization and Cleaning Functions

- Remove leading/trailing spaces, collapse multiple whitespaces, and convert to lowercase;  
- Normalize special or accented characters (`Unidecode`);  
- Retain common alphanumeric characters and basic punctuation;  
- Applied to text fields such as `country`, `province`, `region_1`, `variety`, `winery`, and `taster_name`.


In [69]:

import re, unicodedata
from unidecode import unidecode

def normalize_text_py(s: str) -> str:
    if s is None:
        return None
    # ASCII
    s = unidecode(str(s))
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    # can add more
    return s

normalize_udf = F.udf(normalize_text_py, T.StringType())

# standardize string（keep original，add *_norm for checking）
text_cols = ["country","province","region_1","region_2","variety","winery","taster_name","designation","title"]
for c in text_cols:
    if c in df.columns:
        df = df.withColumn(f"{c}_norm", normalize_udf(F.col(c)))

df.select([F.col(c) for c in df.columns if c.endswith("_norm")]).show(5, truncate=60)


+------------+-----------------+-------------------+-----------------+--------------+-------------------+------------------+----------------------------------+------------------------------------------------------------+
|country_norm|    province_norm|      region_1_norm|    region_2_norm|  variety_norm|        winery_norm|  taster_name_norm|                  designation_norm|                                                  title_norm|
+------------+-----------------+-------------------+-----------------+--------------+-------------------+------------------+----------------------------------+------------------------------------------------------------+
|       italy|sicily & sardinia|               etna|             NULL|   white blend|            nicosia|     kerin o'keefe|                      vulka bianco|                            nicosia 2013 vulka bianco (etna)|
|    portugal|            douro|               NULL|             NULL|portuguese red|quinta dos avidagos|        rog

25/11/09 14:24:45 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv


## 6. Deduplication Strategy

To prevent duplicate entries of the same review, a **fingerprint field** is used:
- First, apply mild normalization to the `description` (lowercasing and removing punctuation);  
- Combine it with `title` and `winery` to generate a SHA2 fingerprint;  
- Perform deduplication based on this fingerprint.



In [70]:

# finger print
df = df.withColumn(
    "desc_fingerprint",
    F.lower(F.regexp_replace(F.col("description"), r"[^A-Za-z0-9]+", " "))
)
df = df.withColumn(
    "row_fingerprint",
    F.sha2(F.concat_ws("||",
                       F.coalesce(F.col("title"), F.lit("")),
                       F.coalesce(F.col("winery"), F.lit("")),
                       F.coalesce(F.col("desc_fingerprint"), F.lit(""))
                      ), 256)
)
before = df.count()
df_dedup = df.dropDuplicates(["row_fingerprint"]).drop("row_fingerprint", "desc_fingerprint")
after = df_dedup.count()
print(f"Deduplicated: {before} -> {after} (removed {before - after})")


25/11/09 14:24:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:24:46 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
[Sta

Deduplicated: 129971 -> 119986 (removed 9985)


## 7. Handling Missing and Abnormal Values

**Price (`price`):**  
- Set clearly invalid values (≤ 0) to `NULL`;  
- Impute missing values using the group median within `country_norm + variety_norm`;  
- Apply mild Winsorization (1% / 99% quantile clipping) to handle outliers.  

**Text Fields:**  
After normalization, any empty strings are uniformly set to `NULL`.

> Note: may modify `groupby_keys` to use combinations like `province_norm + variety_norm` or others to better suit your subsequent analysis.


In [71]:

df_clean = df_dedup

# price value not valid or null
df_clean = df_clean.withColumn(
    "price", F.when(F.col("price").isNotNull() & (F.col("price") <= 0), None).otherwise(F.col("price"))
)

# text NULL
for c in text_cols:
    norm = f"{c}_norm"
    if norm in df_clean.columns:
        df_clean = df_clean.withColumn(norm, F.when(F.trim(F.col(norm)) == "", None).otherwise(F.col(norm)))

# median imputed price within group
groupby_keys = [c for c in ["country_norm","variety_norm"] if c in df_clean.columns]
if groupby_keys:
    med = (
        df_clean
        .groupBy(*groupby_keys)
        .agg(F.expr("percentile_approx(price, 0.5)").alias("med_price"))
    )
    df_clean = (
        df_clean.join(med, on=groupby_keys, how="left")
        .withColumn("price_imputed",
                    F.when(F.col("price").isNull(), F.col("med_price")).otherwise(F.col("price")))
        .drop("med_price")
    )
else:
    df_clean = df_clean.withColumn("price_imputed", F.col("price"))

# 1%–99% winsorization
q = df_clean.approxQuantile("price_imputed", [0.01, 0.99], 0.01)
lower, upper = (q + [None, None])[:2]  # prevent empty list
if lower is not None and upper is not None:
    df_clean = df_clean.withColumn(
        "price_winsor",
        F.when(F.col("price_imputed") < lower, F.lit(lower))
         .when(F.col("price_imputed") > upper, F.lit(upper))
         .otherwise(F.col("price_imputed"))
    )
else:
    df_clean = df_clean.withColumn("price_winsor", F.col("price_imputed"))

print("Rows after cleaning:", df_clean.count())


25/11/09 14:24:50 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:24:53 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
[Sta

Rows after cleaning: 119986


## 8. Variety Normalization Mapping (Synonym Merging)

For analytical convenience, common grape varieties can be manually merged  
(e.g., *pinot grigio* and *pinot gris*).  
The example below provides an extensible mapping table.  




In [72]:

from pyspark.sql import Row

VARIETY_MAP = {
    "cabernet sauvignon": ["cabernet sauv.", "cab sauv", "cabernet"],
    "sauvignon blanc": ["sauv. blanc"],
    "pinot noir": ["spatburgunder"],  # German variant (after unidecode → spatburgunder)
    "pinot grigio": ["pinot gris"],
    "syrah": ["shiraz"],
    "grenache": ["garnacha"],
    "tempranillo": [],
    "nebbiolo": [],
    "chardonnay": [],
    "riesling": [],
    "merlot": [],
    "malbec": [],
    "sangiovese": [],
    "chenin blanc": ["steen"],
    "viognier": [],
    "gamay": [],
    "gruner veltliner": ["gruener veltliner", "gruner veltliner"],
}

# Reverse index: alias → standard name (matched after normalization)
alias_to_canonical = {}
for canon, aliases in VARIETY_MAP.items():
    alias_to_canonical[canon] = canon
    for a in aliases:
        alias_to_canonical[a] = canon

broadcast_map = spark.sparkContext.broadcast(alias_to_canonical)

@F.udf(T.StringType())
def map_variety_norm(v):
    if v is None:
        return None
    v = v.strip().lower()
    # direct match or alias mapping
    if v in broadcast_map.value:
        return broadcast_map.value[v]
    # simple rule: remove trailing “blend”, etc. (handle with caution)
    if v.endswith(" blend"):
        return v.replace(" blend", "").strip()
    return v

if "variety_norm" in df_clean.columns:
    df_clean = df_clean.withColumn("variety_canonical", map_variety_norm(F.col("variety_norm")))
else:
    df_clean = df_clean.withColumn("variety_canonical", F.col("variety"))

df_clean.select("variety","variety_norm","variety_canonical").show(8, truncate=30)


25/11/09 14:24:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
[Stage 36:>                                                         (0 + 1) / 1]

+------------------+------------------+------------------+
|           variety|      variety_norm| variety_canonical|
+------------------+------------------+------------------+
|              Rosé|              rose|              rose|
|         Zinfandel|         zinfandel|         zinfandel|
|        Pinot Noir|        pinot noir|        pinot noir|
|        Pinot Noir|        pinot noir|        pinot noir|
|   Sauvignon Blanc|   sauvignon blanc|   sauvignon blanc|
|        Chardonnay|        chardonnay|        chardonnay|
|        Chardonnay|        chardonnay|        chardonnay|
|Cabernet Sauvignon|cabernet sauvignon|cabernet sauvignon|
+------------------+------------------+------------------+
only showing top 8 rows



## 9. Description Text Cleaning and Tokenization → TF-IDF

- Convert `description` to lowercase and remove non-English characters;  
- Tokenize using `RegexTokenizer`;  
- Remove stopwords with `StopWordsRemover`;  
- Generate sparse `tfidf` vectors using `CountVectorizer` + `IDF`.  

> Note: If local memory is limited, you can set `SAMPLE_FRACTION < 1.0` in the configuration section for sampled debugging.



In [73]:

from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF

# configurable sampling (very useful for debugging)
SAMPLE_FRACTION = float(os.environ.get("SAMPLE_FRACTION", "1.0"))
df_text = df_clean
if SAMPLE_FRACTION < 1.0:
    df_text = df_text.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=42)

# normalize description text
df_text = df_text.withColumn("description_clean",
                             F.lower(F.regexp_replace(F.col("description"), r"[^A-Za-z\s]+", " ")))

tokenizer = RegexTokenizer(inputCol="description_clean", outputCol="tokens_raw", pattern="\W+")
remover = StopWordsRemover(inputCol="tokens_raw", outputCol="tokens")
cv = CountVectorizer(inputCol="tokens", outputCol="tf", minDF=10)  # Adjust up or down depending on available memory
idf = IDF(inputCol="tf", outputCol="tfidf", minDocFreq=10)

text_pipe = Pipeline(stages=[tokenizer, remover, cv, idf])
text_model = text_pipe.fit(df_text)
df_text_feats = text_model.transform(df_text)

df_text_feats.select("description", "tokens").show(5, truncate=80)


25/11/09 14:25:28 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:25:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/1

+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|                                                                     description|                                                                          tokens|
+--------------------------------------------------------------------------------+--------------------------------------------------------------------------------+
|In its Provence skittle bottle, this wine is a bright mouthful that is as eas...|[provence, skittle, bottle, wine, bright, mouthful, easy, palate, pale, color...|
|Thick and rich, with a menthol-like streak in the background, this 89% Zin is...|[thick, rich, menthol, like, streak, background, zin, complemented, wealth, m...|
|The fragrant mix of flowers and fruit introduces a wine already moving into s...|[fragrant, mix, flowers, fruit, introduces, wine, already, moving, secondary,...|
|There are a lot

## 10. Structured Feature Encoding (Indexing / One-Hot)

- Use `StringIndexer` to index fields such as `country_norm` and `variety_canonical`;  
- Apply `OneHotEncoder` to generate one-hot vectors;  
- Combine them with numerical features (`price_winsor`, `points`) into a single feature vector for subsequent modeling.


In [74]:

from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

categorical_cols = [c for c in ["country_norm","variety_canonical","province_norm"] if c in df_text_feats.columns]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in categorical_cols]
encoders = [OneHotEncoder(inputCols=[f"{c}_idx"], outputCols=[f"{c}_oh"]) for c in categorical_cols]

numeric_cols = [c for c in ["points","price_winsor"] if c in df_text_feats.columns]

assembler = VectorAssembler(
    inputCols=[*numeric_cols, *[f"{c}_oh" for c in categorical_cols], "tfidf"],
    outputCol="features"
)

struct_pipe = Pipeline(stages=[*indexers, *encoders, assembler])
struct_model = struct_pipe.fit(df_text_feats)
df_ready = struct_model.transform(df_text_feats)

df_ready.select("points","price_winsor", *[f"{c}_oh" for c in categorical_cols], "features").show(3, truncate=80)
df_ready.printSchema()
print(df_text_feats.columns)

25/11/09 14:25:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:25:44 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/1

+------+------------+---------------+--------------------+----------------+--------------------------------------------------------------------------------+
|points|price_winsor|country_norm_oh|variety_canonical_oh|province_norm_oh|                                                                        features|
+------+------------+---------------+--------------------+----------------+--------------------------------------------------------------------------------+
|    90|        17.0|     (43,[],[])|   (698,[215],[1.0])|     (425,[],[])|(9546,[0,1,260,1168,1170,1172,1188,1198,1206,1208,1216,1222,1266,1325,1340,13...|
|    89|        28.0|     (43,[],[])|     (698,[1],[1.0])|     (425,[],[])|(9546,[0,1,46,1168,1169,1171,1173,1195,1233,1236,1244,1256,1286,1300,1321,137...|
|    88|        36.0|     (43,[],[])|     (698,[1],[1.0])|     (425,[],[])|(9546,[0,1,46,1168,1169,1171,1172,1176,1181,1182,1189,1214,1219,1226,1262,129...|
+------+------------+---------------+--------------------+

## 11. Persistence and Output

- `outputs/wine_clean.parquet/`: cleaned base dataset;  
- `outputs/wine_ml_ready.parquet/`: model-ready dataset containing `tokens`, `tfidf`, and `features`;  
- `outputs/models/text_pipeline/` and `outputs/models/struct_pipeline/`: reusable preprocessing models.  

> If running on a cluster or distributed file system, can replace `OUTPUT_DIR` with an HDFS or S3 path.



In [75]:
from pyspark import StorageLevel
from pyspark.ml.feature import PCA, VectorAssembler
from pyspark.sql import functions as F

# ========= 1) 先保存干净表（你原来的逻辑） =========
core_cols = [c for c in [
    "country","country_norm","province","province_norm","region_1","region_1_norm","region_2","region_2_norm",
    "variety","variety_norm","variety_canonical","winery","winery_norm",
    "designation","designation_norm","title","title_norm",
    "taster_name","taster_name_norm","taster_twitter_handle",
    "description","points","price","price_imputed","price_winsor"
] if c in df_clean.columns]

record_count = df_clean.count()
(df_clean
    .select(*core_cols)
    .write.mode("overwrite")
    .option("compression", "snappy")
    .option("parquet.block.size", 64 * 1024 * 1024)
    .parquet(str(OUTPUT_DIR / "wine_clean.parquet"))
)
print(f"✅ Saved partitioned clean dataset ({record_count:,} rows) to:", (OUTPUT_DIR / 'wine_clean.parquet').resolve())

# ========= 2) 对 TF-IDF 做 PCA 降维 =========
# 说明：这里假设你已经有 df_text_feats（text_model.transform 之后的表：包含 'tfidf' 列）
#       如果没有，可先：df_text_feats = text_model.transform(df_text)

if "tfidf" not in df_text_feats.columns:
    raise ValueError("未找到 'tfidf' 列，请先运行 text_model.transform 得到 df_text_feats。")

# 自动选择合适的 k：不超过 500，不超过实际 tfidf 维度，至少 100
tfidf_dim = df_text_feats.select("tfidf").first()["tfidf"].size
k = max(100, min(500, tfidf_dim))
print(f"🔄 Performing PCA reduction on TF-IDF features ... (orig={tfidf_dim}, k={k})")

if "tfidf_pca" in df_text_feats.columns:
    df_text_feats = df_text_feats.drop("tfidf_pca")
pca = PCA(k=k, inputCol="tfidf", outputCol="tfidf_pca")
pca_model = pca.fit(df_text_feats)
df_text_feats = pca_model.transform(df_text_feats)
print(f"✅ PCA reduction complete (TF-IDF → {k} dims)")

# ========= 3) 生成 OneHot（如果你已有 struct_model，就用它；否则临时做一遍） =========
need_cols = {"country_norm_oh", "province_norm_oh", "variety_canonical_oh"}
have_cols = set(df_text_feats.columns)

if not need_cols.issubset(have_cols):
    # 用你已有的 struct_model（StringIndexer + OneHotEncoder）来补齐
    df_ready_feats = struct_model.transform(df_text_feats)
else:
    df_ready_feats = df_text_feats

# ========= 4) 安全两步拼接成 features =========
assembler_struct = VectorAssembler(
    inputCols=["points", "price_winsor", "country_norm_oh", "province_norm_oh", "variety_canonical_oh"],
    outputCol="struct_features",
    handleInvalid="keep"
)
df_struct = assembler_struct.transform(df_ready_feats)
if "features" in df_struct.columns:          # ← rm features 
    df_struct = df_struct.drop("features")
assembler_full = VectorAssembler(
    inputCols=["struct_features", "tfidf_pca"],
    outputCol="features",
    handleInvalid="keep"
)
df_ready = assembler_full.transform(df_struct)
print("✅ Feature assembly complete.")

# ========= 5) 写出建模数据（压缩 + 控制分区/块大小，避免内存尖峰） =========
df_ready_to_save = df_ready.select(
    *[c for c in core_cols if c in df_ready.columns],  # 需要的话保留核心列
    "tfidf_pca", "features"
)

df_ready_to_save.persist(StorageLevel.MEMORY_AND_DISK)
(df_ready_to_save
    .repartition(8)
    .write.mode("overwrite")
    .option("compression", "snappy")
    .option("parquet.block.size", 64 * 1024 * 1024)
    .parquet(str(OUTPUT_DIR / "wine_ml_ready.parquet"))
)
print(f"✅ Saved ML-ready dataset to:", (OUTPUT_DIR / "wine_ml_ready.parquet").resolve())
df_ready_to_save.unpersist()

# ========= 6) 保存模型 =========
text_model.write().overwrite().save(str(OUTPUT_DIR / "models" / "text_pipeline"))
struct_model.write().overwrite().save(str(OUTPUT_DIR / "models" / "struct_pipeline"))
pca_model.write().overwrite().save(str(OUTPUT_DIR / "models" / "pca_model"))

print("✅ Models:", (OUTPUT_DIR / "models").resolve())


25/11/09 14:26:04 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:26:06 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/1

✅ Saved partitioned clean dataset (119,986 rows) to: /home/senluowsl/ds410/project/archive/outputs/wine_clean.parquet


25/11/09 14:26:15 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv


🔄 Performing PCA reduction on TF-IDF features ... (orig=8378, k=500)


✅ PCA reduction complete (TF-IDF → 500 dims)
✅ Feature assembly complete.


25/11/09 14:27:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:27:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/1

✅ Saved ML-ready dataset to: /home/senluowsl/ds410/project/archive/outputs/wine_ml_ready.parquet


25/11/09 14:28:14 WARN TaskSetManager: Stage 179 contains a task of very large size (33413 KiB). The maximum recommended task size is 1000 KiB.
[Stage 181:>                                                        (0 + 1) / 1]

✅ Models: /home/senluowsl/ds410/project/archive/outputs/models


## 12. Quick Sanity Check / EDA


In [76]:

# report of missing values (top 12 columns)
null_report(df_clean)[:12]


25/11/09 14:29:33 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:29:35 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/1

[('region_2', 73218, 0.6102),
 ('region_2_norm', 73218, 0.6102),
 ('designation', 34545, 0.2879),
 ('designation_norm', 34545, 0.2879),
 ('taster_twitter_handle', 29446, 0.2454),
 ('taster_name', 24917, 0.2077),
 ('taster_name_norm', 24917, 0.2077),
 ('region_1', 19560, 0.163),
 ('region_1_norm', 19560, 0.163),
 ('price', 8395, 0.07),
 ('country_norm', 59, 0.0005),
 ('country', 59, 0.0005)]

In [78]:

# Top 10 
if "country_norm" in df_clean.columns:
    display(df_clean.groupBy("country_norm").count().orderBy(F.desc("count")).limit(10))
if "variety_canonical" in df_clean.columns:
    display(df_clean.groupBy("variety_canonical").count().orderBy(F.desc("count")).limit(10))
df_clean.groupBy("country_norm").count().orderBy(F.desc("count")).show(10)
df_clean.groupBy("variety_canonical").count().orderBy(F.desc("count")).show(10)

DataFrame[country_norm: string, count: bigint]

DataFrame[variety_canonical: string, count: bigint]

25/11/09 14:31:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv
25/11/09 14:31:41 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
 Schema: _c0, country, description, designation, points, price, province, region_1, region_2, taster_name, taster_twitter_handle, title, variety, winery
Expected: _c0 but found: 
CSV file: file:///home/senluowsl/ds410/project/archive/data/winemag-data-130k-v2.csv


+------------+-----+
|country_norm|count|
+------------+-----+
|          us|50456|
|      france|20353|
|       italy|17940|
|       spain| 6116|
|    portugal| 5256|
|       chile| 4184|
|   argentina| 3544|
|     austria| 3034|
|   australia| 2196|
|     germany| 1992|
+------------+-----+
only showing top 10 rows



[Stage 211:>                                                        (0 + 1) / 1]

+------------------+-----+
| variety_canonical|count|
+------------------+-----+
|        pinot noir|12344|
|        chardonnay|10868|
|cabernet sauvignon| 8850|
|               red| 8243|
|bordeaux-style red| 6471|
|          riesling| 4773|
|             syrah| 4617|
|   sauvignon blanc| 4575|
|              rose| 3220|
|            merlot| 2896|
+------------------+-----+
only showing top 10 rows



## 13. Appendix: How to Reuse the Same Preprocessing Pipeline on New Data

When you have a new `DataFrame` (with the same or similar column structure as the input used in this notebook):

1. Perform the same cleaning steps as in this notebook (normalization, deduplication, missing value handling, etc.);  
2. Load the saved `text_pipeline` and `struct_pipeline`;  
3. Simply call `transform(new_df)` to obtain the `tfidf` and `features` outputs.

Example:


In [ ]:

from pyspark.ml import PipelineModel

loaded_text = PipelineModel.load(str(OUTPUT_DIR / "models" / "text_pipeline"))
loaded_struct = PipelineModel.load(str(OUTPUT_DIR / "models" / "struct_pipeline"))

# suppose new_df_clean and df_clean have identical lists
# new_df_text = loaded_text.transform(new_df_clean)
# new_df_ready = loaded_struct.transform(new_df_text)
# new_df_ready.select("features").show(3)
print("Models loaded. Use `loaded_text.transform(df)` then `loaded_struct.transform(df)` on new data.")



## 14. end session


In [1]:

spark.stop() 


NameError: name 'spark' is not defined

In [4]:
from pyspark.sql import SparkSession

# 启动 Spark 会话
spark = SparkSession.builder.appName("ParquetToCSV").getOrCreate()

# 读取 parquet 数据
df_clean = spark.read.parquet("/home/senluowsl/ds410/project/archive/outputs/wine_clean.parquet")

# 写出为 CSV（带 header）
(df_clean
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv("/home/senluowsl/ds410/project/archive/outputs/wine_clean_csv")
)

print("CSV 已保存到目录: /home/senluowsl/ds410/project/archive/outputs/wine_clean_csv")


25/11/09 14:45:09 WARN Utils: Your hostname, DESKTOP-PQOC1I8 resolves to a loopback address: 127.0.1.1; using 172.28.233.232 instead (on interface eth0)
25/11/09 14:45:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/09 14:45:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 1:>                                                          (0 + 1) / 1]

✅ CSV 已保存到目录: /home/senluowsl/ds410/project/archive/outputs/wine_clean_csv
